# 02a - Channel overview

Chooses which images form the analysis cube and inspects their relationships
before any modelling.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [1]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
print("Raw data :", config.raw_dir)
print("Outputs  :", config.processed_dir)

Raw data : C:\Users\amanb\OneDrive\Desktop\dissertation\Fossil Fly\data\raw
Outputs  : C:\Users\amanb\OneDrive\Desktop\dissertation\Fossil Fly\data\processed


## Choose the analysis channels

The most common pixel grid becomes the analysis grid, and the polarity that
dominates it becomes the analysis polarity. Everything else is treated as a
secondary acquisition. Both choices can be pinned in the config.

In [2]:
from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)

print(selection.describe())
for key, label in zip(selection.keys, selection.labels):
    print(f"  {label:22s} <- {key}")

5 channels @ 640x640 (Neg mode); 2 secondary image(s)
  m/z 62.96 (Neg)        <- Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_62.96198±0.01675u
  m/z 78.95 (Neg)        <- Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_78.94747±0.01675u
  m/z 96.94 (Neg)        <- Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_96.93916±0.02297u
  m/z 103.91 (Neg)       <- Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_103.91326±0.02297u
  m/z 123.95 (Neg)       <- Fly_Neg_C60_30pA_10x10x64pix_400sh-35V_123.94783±0.02297u


In [3]:
cube = build_cube(images, selection)
print("analysis cube:", cube.shape)

analysis cube: (640, 640, 5)


## Channel series and composites

In [4]:
from src.viz import overview

overview.plot_channel_series(
    images, selection, config.figure_path("02a_channel_series.png"), config
)
overview.plot_rgb_composites(
    images, selection, config.figure_path("02a_rgb_composites.png"), config
)
print("figures written")

figures written


## Inter-channel correlation

Strongly correlated channels carry redundant spatial information, which is
what makes the decomposition in 02b worthwhile.

In [5]:
import pandas as pd

matrix = overview.channel_correlation_matrix(cube)
overview.plot_correlation_matrix(
    matrix, selection.labels, config.figure_path("02a_correlation_matrix.png")
)
pd.DataFrame(matrix, index=selection.labels, columns=selection.labels).round(3)

,m/z 62.96 (Neg),m/z 78.95 (Neg),m/z 96.94 (Neg),m/z 103.91 (Neg),m/z 123.95 (Neg)
m/z 62.96 (Neg),1.000,0.841,0.000,0.008,0.027
m/z 78.95 (Neg),0.841,1.000,0.039,0.048,0.048
m/z 96.94 (Neg),0.000,0.039,1.000,0.035,0.032
m/z 103.91 (Neg),0.008,0.048,0.035,1.000,0.178
m/z 123.95 (Neg),0.027,0.048,0.032,0.178,1.000


## Compare the acquisitions

In [6]:
if selection.secondary_keys:
    overview.plot_polarity_comparison(
        images, selection, config.figure_path("02a_polarity_comparison.png")
    )
    overview.plot_mean_comparison(
        images, selection, config.figure_path("02a_polarity_mean_comparison.png")
    )
    print("figures written")
else:
    print("No secondary acquisition in this dataset.")

figures written


Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage overview
```